In [1]:
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split, GridSearchCV

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.metrics import accuracy_score

In [2]:
df = pd.read_csv("./data/titanic_procesado.csv")

print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
print(f"Valores nulos: {df.isnull().sum().sum()}")

df.head()

Filas: 891
Columnas: 8
Valores nulos: 0


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,1.0,1.0,0.433152,0.125,0.0,0.368146,1.0
1,1,0.0,0.0,0.579431,0.125,0.0,0.615097,0.0
2,1,1.0,0.0,0.462346,0.000,0.0,0.438286,1.0
3,1,0.0,0.0,0.563806,0.125,0.0,0.595112,1.0
4,0,1.0,1.0,0.563806,0.000,0.0,0.448347,1.0


In [3]:
X = df.drop(columns=["Survived"])
y = df["Survived"]

print("Dimensiones de X:", X.shape)
print("Dimensiones de y:", y.shape)

X.head()

Dimensiones de X: (891, 7)
Dimensiones de y: (891,)


,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,1.0,1.0,0.433152,0.125,0.0,0.368146,1.0
1,0.0,0.0,0.579431,0.125,0.0,0.615097,0.0
2,1.0,0.0,0.462346,0.000,0.0,0.438286,1.0
3,0.0,0.0,0.563806,0.125,0.0,0.595112,1.0
4,1.0,1.0,0.563806,0.000,0.0,0.448347,1.0


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

X_train = X_train.to_numpy()
X_test = X_test.to_numpy()
y_train = y_train.to_numpy()
y_test = y_test.to_numpy()

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (712, 7)
X_test: (179, 7)
y_train: (712,)
y_test: (179,)


In [5]:
modelos = {
    "Regresión Logística": {
        "modelo": LogisticRegression(random_state=42),
        "parametros": {
            "C": [0.01, 0.1, 1, 10, 100],
            "penalty": ["l1", "l2"],
            "solver": ["liblinear", "saga"],
            "max_iter": [100, 500, 1000]
        }
    },

    "Clasificador de Vectores de Soporte": {
        "modelo": SVC(),
        "parametros": {
            "kernel": ["linear", "poly", "rbf", "sigmoid"],
            "C": [0.1, 1, 10]
        }
    },

    "Clasificador de Árbol de Decisión": {
        "modelo": DecisionTreeClassifier(random_state=42),
        "parametros": {
            "splitter": ["best", "random"],
            "max_depth": [None, 1, 2, 3, 4]
        }
    },

    "Clasificador de Bosques Aleatorios": {
        "modelo": RandomForestClassifier(random_state=42),
        "parametros": {
            "n_estimators": [10, 100],
            "max_depth": [None, 1, 2, 3, 4],
            "max_features": ["sqrt", "log2", None]
        }
    },

    "Clasificador de Gradient Boosting": {
        "modelo": GradientBoostingClassifier(random_state=42),
        "parametros": {
            "n_estimators": [10, 100],
            "max_depth": [None, 1, 2, 3, 4]
        }
    },

    "Clasificador AdaBoost": {
        "modelo": AdaBoostClassifier(random_state=42),
        "parametros": {
            "n_estimators": [10, 100]
        }
    },

    "Clasificador K-Nearest Neighbors": {
        "modelo": KNeighborsClassifier(),
        "parametros": {
            "n_neighbors": [3, 5, 7]
        }
    },

    "Clasificador XGBoost": {
        "modelo": XGBClassifier(
            random_state=42,
            eval_metric="logloss"
        ),
        "parametros": {
            "n_estimators": [10, 100],
            "max_depth": [1, 2, 3, 6]
        }
    },

    "Clasificador LGBM": {
        "modelo": LGBMClassifier(
            random_state=42,
            verbosity=-1
        ),
        "parametros": {
            "n_estimators": [10, 100],
            "max_depth": [-1, 1, 2, 3],
            "learning_rate": [0.1, 0.2, 0.3]
        }
    },

    "GaussianNB": {
        "modelo": GaussianNB(),
        "parametros": {}
    },

    "Clasificador Naive Bayes": {
        "modelo": BernoulliNB(),
        "parametros": {
            "alpha": [0.1, 1.0, 10.0]
        }
    }
}

In [6]:
puntajes_modelos = []
mejor_precision = 0
mejor_estimador = None
mejor_modelo = None
estimadores = {}

for nombre, info_modelo in modelos.items():
    print(f"Entrenando: {nombre}")

    grid_search = GridSearchCV(
        estimator=info_modelo["modelo"],
        param_grid=info_modelo["parametros"],
        cv=5,
        scoring="accuracy",
        verbose=0,
        n_jobs=-1
    )

    grid_search.fit(X_train, y_train)

    y_pred = grid_search.predict(X_test)
    precision = accuracy_score(y_test, y_pred)

    puntajes_modelos.append({
        "Modelo": nombre,
        "Precisión": precision,
        "Mejores hiperparámetros": grid_search.best_params_
    })

    estimadores[nombre] = grid_search.best_estimator_

    if precision > mejor_precision:
        mejor_modelo = nombre
        mejor_precision = precision
        mejor_estimador = grid_search.best_estimator_

    print(f"Precisión: {precision:.4f}")

Entrenando: Regresión Logística


C:\Users\ingen\Downloads\aprendizaje_supervisado\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


Precisión: 0.7933
Entrenando: Clasificador de Vectores de Soporte
Precisión: 0.7989
Entrenando: Clasificador de Árbol de Decisión
Precisión: 0.7989
Entrenando: Clasificador de Bosques Aleatorios
Precisión: 0.7989
Entrenando: Clasificador de Gradient Boosting
Precisión: 0.8156
Entrenando: Clasificador AdaBoost
Precisión: 0.7933
Entrenando: Clasificador K-Nearest Neighbors
Precisión: 0.8156
Entrenando: Clasificador XGBoost
Precisión: 0.8156
Entrenando: Clasificador LGBM
Precisión: 0.8156
Entrenando: GaussianNB
Precisión: 0.7654
Entrenando: Clasificador Naive Bayes
Precisión: 0.7877


In [7]:
metricas = (
    pd.DataFrame(puntajes_modelos)
    .sort_values("Precisión", ascending=False)
    .reset_index(drop=True)
)

print("Rendimiento de los modelos de clasificación")
display(metricas)

print("---------------------------------------------------")
print("MEJOR MODELO DE CLASIFICACIÓN")
print(f"Modelo: {mejor_modelo}")
print(f"Precisión: {mejor_precision:.4f}")
print(f"Estimador: {mejor_estimador}")

Rendimiento de los modelos de clasificación


,Modelo,Precisión,Mejores hiperparámetros
0,Clasificador LGBM,0.815642,"{'learning_rate': 0.1, 'max_depth': 3, 'n_esti..."
1,Clasificador K-Nearest Neighbors,0.815642,{'n_neighbors': 7}
2,Clasificador de Gradient Boosting,0.815642,"{'max_depth': 2, 'n_estimators': 100}"
3,Clasificador XGBoost,0.815642,"{'max_depth': 3, 'n_estimators': 10}"
4,Clasificador de Bosques Aleatorios,0.798883,"{'max_depth': 3, 'max_features': None, 'n_esti..."
5,Clasificador de Vectores de Soporte,0.798883,"{'C': 1, 'kernel': 'rbf'}"
6,Clasificador de Árbol de Decisión,0.798883,"{'max_depth': 3, 'splitter': 'best'}"
7,Regresión Logística,0.793296,"{'C': 1, 'max_iter': 100, 'penalty': 'l2', 'so..."
8,Clasificador AdaBoost,0.793296,{'n_estimators': 100}
9,Clasificador Naive Bayes,0.787709,{'alpha': 1.0}


---------------------------------------------------
MEJOR MODELO DE CLASIFICACIÓN
Modelo: Clasificador de Gradient Boosting
Precisión: 0.8156
Estimador: GradientBoostingClassifier(max_depth=2, random_state=42)


In [8]:
nuevo_pasajero = X_test[0].reshape(1, -1)

prediccion = mejor_estimador.predict(nuevo_pasajero)[0]
valor_real = y_test[0]

resultado = "sobrevivió" if prediccion == 1 else "no sobrevivió"

print(f"Predicción del modelo: {resultado}")
print(f"Valor real: {valor_real}")

Predicción del modelo: no sobrevivió
Valor real: 1


In [9]:
with open("modelo.pkl", "wb") as archivo_estimador:
    pickle.dump(mejor_estimador, archivo_estimador)

print("Modelo guardado correctamente en modelo.pkl")

Modelo guardado correctamente en modelo.pkl


In [10]:
with open("modelo.pkl", "rb") as archivo_estimador:
    modelo_cargado = pickle.load(archivo_estimador)

prediccion_prueba = modelo_cargado.predict(
    X_test[0].reshape(1, -1)
)

print("Predicción del modelo cargado:", prediccion_prueba)

Predicción del modelo cargado: [0]
